In [ ]:
import numpy as np

# **NM WaterStar**

In [ ]:
import pandas as pd
import os
from google.colab import drive

# Mount Google Drive to access files
drive.mount('/content/drive')

# Define the base path relative to the mounted Google Drive
# This path is set to reflect the common structure 'Atosi/pw_analysis' found in other file references.
BASE_PATH = "/content/drive/MyDrive/Atosi/pw_analysis"

In [ ]:
import os

LOCAL_DATA_DIR = '/content/data_clean'
# Ensure the local data directory exists, if necessary
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

In [ ]:
df = pd.read_csv(os.path.join(LOCAL_DATA_DIR, 'all_quarter_townships_quantity_data.csv'))
display(df.head())

In [ ]:
df = pd.DataFrame(all_data)
df.head()

In [ ]:
print(f"Dataset shape (rows, columns): {df.shape}")

In [ ]:
df.info()

A helper function clean_columns defined to standardize column names by converting them to lowercase, stripping whitespace, and replacing spaces with underscores. Although the clean_columns function was defined, the column renaming was directly applied to the DataFrame first by converting all column names to lowercase and then using a rename dictionary to give more descriptive names like geo_id, qty_5yr, tds, etc.

In [ ]:
def clean_columns(df):
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )
    return df

In [ ]:
df.head()

In [ ]:
df.columns = df.columns.str.lower()
df.head()

In [ ]:
df = df.rename(columns={
    "id": "geo_id",
    "quantitylastfiveyears": "qty_5yr",
    "quantitylastyear": "qty_1yr",
    "quantitylast365days": "qty_365d",
    "quantitywellcount": "qty_well_count",
    "waterquality_tds": "tds",
    "qualitywellcount": "qual_well_count",
    "qualitysamplecount": "qual_sample_count"
})

**units:**
1.   qnt - bbl
2.   tds - mg/L

In [ ]:
df.describe()

Several columns (including qty_5yr, qty_1yr, tds, qual_sample_count, qty_well_count, waterquality_turbidity, waterquality_temp, waterquality_na, waterquality_cl) were converted to numeric types using pd.to_numeric with errors='coerce' to handle any non-numeric values gracefully. Following this, rows with NaN values in qty_5yr, qty_1yr, and tds were dropped to ensure data quality for these key metrics.

In [ ]:
df["tds"] = pd.to_numeric(df["tds"], errors="coerce")

print(df["tds"].head())

In [ ]:
df.dtypes

In [ ]:
import numpy as np

cols = ["qty_5yr", "qty_1yr", "tds", "qual_sample_count", "qty_well_count","waterquality_turbidity","waterquality_temp", "waterquality_na", "waterquality_cl" ]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df = df[df["qty_5yr"].notna()]
df = df[df["qty_1yr"].notna()]
df = df[df["tds"].notna()]

In [ ]:
df.head()

**Feature Engineering:** Several new features were created:

log_qty: Natural logarithm of qty_5yr (plus one, using np.log1p).
log_tds: Natural logarithm of tds (plus one, using np.log1p).
supply_intensity: Calculated as qty_5yr divided by qty_well_count.

In [ ]:
df["log_qty"] = np.log1p(df["qty_5yr"])
df["log_tds"] = np.log1p(df["tds"])

In [ ]:
df["supply_intensity"] = df["qty_5yr"] / df["qty_well_count"]

In [ ]:
df.to_csv(os.path.join(BASE_PATH, 'data_clean', 'nm_clean.csv'), index=False)

Purpose: Merge quantity time-series with quality data into df_final.csv, after running statistical exploration of the time-series features first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Atosi/pw_analysis"
os.chdir(BASE_PATH)

print("Current path:", os.getcwd())

In [ ]:
import pandas as pd

file_path = os.path.join(BASE_PATH, 'data_clean', 'all_quarter_townships_quantity_data.csv')

try:
    df_allquarter_qnt = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}.")
    display(df_allquarter_qnt.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
def clean_columns(df_allquarter_qnt):
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )
    return df

In [ ]:
df_allquarter_qnt.columns = df_allquarter_qnt.columns.str.lower()
df_allquarter_qnt.head()

In [ ]:
df_allquarter_qnt = df_allquarter_qnt.rename(columns={'quarter township': 'geo_id', 'value': 'volume'})

print("Columns renamed successfully. Displaying first 5 rows with updated column names:")
display(df_allquarter_qnt.head()

In [ ]:
import numpy as np
import pandas as pd

# Convert 'date' column to datetime objects
df_allquarter_qnt["date"] = pd.to_datetime(df_allquarter_qnt["date"])

# Create df_yearly by aggregating volume per geo_id and year
df_yearly = df_allquarter_qnt.groupby(["geo_id", df_allquarter_qnt["date"].dt.year])["volume"].sum().reset_index()
df_yearly.rename(columns={'date': 'date_year'}, inplace=True)

def compute_trend(group):
    # group will have 'geo_id', 'date_year' (which is actually year), 'volume'
    group = group.sort_values("date_year") # Sort by year

    if len(group) < 3:
        return np.nan

    x = group["date_year"].values # Use actual years as x values, not just arange
    y = group["volume"].values

    # Check for constant x values, which would cause an error in polyfit
    if len(np.unique(x)) < 2: # Need at least 2 distinct points for a line
        return np.nan

    return np.polyfit(x, y, 1)[0]

trend_df = df_yearly.groupby("geo_id").apply(compute_trend).reset_index()
trend_df.columns = ["geo_id", "ts_trend"]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(trend_df['ts_trend'].dropna(), bins=50, kde=True)
plt.title('Distribution of Yearly Volume Trend (ts_trend)')
plt.xlabel('Yearly Volume Trend')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()
# average yearly change in water volume for each geo_id

In [ ]:
#the standard deviation of 'volume' for each geo_id from df_yearly
vol_df = df_yearly.groupby("geo_id")["volume"].std().reset_index()
vol_df.columns = ["geo_id", "ts_volatility"]

In [ ]:
def recent_growth(group):
    group = group.sort_values("date_year")

    if len(group) < 2:
        return np.nan

    return group["volume"].iloc[-1] / group["volume"].iloc[-2]

growth_df = df_yearly.groupby("geo_id").apply(recent_growth).reset_index()
growth_df.columns = ["geo_id", "recent_growth"]

In [ ]:
file_path = os.path.join(BASE_PATH, 'data_clean', 'nm_clean.csv')

try:
    df_qnt_qual = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}.")
    display(df_qnt_qual.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
df_qnt_qual.shape

In [ ]:
print("Current columns in df_qnt_qual:")
print(df_qnt_qual.columns.tolist())

columns_to_drop = ['log_qty', 'log_tds', 'confidence', 'supply_intensity']

# Filter out columns that don't exist in the DataFrame
existing_columns_to_drop = [col for col in columns_to_drop if col in df_qnt_qual.columns]

if existing_columns_to_drop:
    df_qnt_qual = df_qnt_qual.drop(columns=existing_columns_to_drop)
    print(f"Successfully dropped columns: {existing_columns_to_drop}")
else:
    print("None of the specified columns to drop were found in the DataFrame.")

print("Displaying first 5 rows with updated columns:")
display(df_qnt_qual.head())

In [ ]:
df_core = df_qnt_qual[[
    "geo_id",
    "qty_5yr",
    "qty_1yr",
    "qty_365d",
    "tds",
    "qual_sample_count"
]].copy()

In [ ]:
df_core.head()

In [ ]:
df_core.isna().sum()

In [ ]:
(df_core.isna().mean() * 100).sort_values(ascending=False)

In [ ]:
df_core.shape

In [ ]:
df_core["qty_5yr"].describe()

In [ ]:
import matplotlib.pyplot as plt

df_core["qty_5yr"].hist(bins=50)
plt.title("Qty 5yr Distribution")
plt.show()

In [ ]:
df_core["tds"].hist(bins=50)
plt.title("TDS Distribution")
plt.show()

In [ ]:
summary = pd.DataFrame({
    "missing_%": df_core.isna().mean() * 100,
    "zeros_%": (df_core == 0).mean() * 100
})

summary

In [ ]:
df_core["tds"].describe()

In [ ]:
df_core["log_tds"] = np.log1p(df_core["tds"])

In [ ]:
df_core["log_tds"].hist(bins=50)
plt.title("Log TDS Distribution")
plt.show()

**Quanity of PW in last 5 years**

In [ ]:
df_core["qty_5yr"].describe()

In [ ]:
df_core["qty_5yr"].hist(bins=50)
plt.title("Last 5 years Distribution")
plt.show()

In [ ]:
df_core["log_qty_5yr"] = np.log1p(df_core["qty_5yr"])

In [ ]:
df_core["log_qty_5yr"].hist(bins=50)
plt.title("Log Last 5 years Distribution")
plt.show()

In [ ]:
df_core["qty_5yr"].describe()

In [ ]:
plt.scatter(df_core['log_qty_5yr'], df_core['log_tds']) #

x axis: natural logarithm of the total water volume in the last 5 years for each geo_id

y-axis:the natural logarithm of the Total Dissolved Solids (TDS) for each geo_id

**Merge both Quantity + Quality Datasets**

In [ ]:
df_final = df_core.merge(trend_df, on="geo_id", how="left")
df_final = df_final.merge(vol_df, on="geo_id", how="left")
df_final = df_final.merge(growth_df, on="geo_id", how="left")

In [ ]:
df_final.head()

In [ ]:
df_final[["ts_trend", "ts_volatility"]].describe()

In [ ]:
df_final["ts_trend_norm"] = df_final["ts_trend"] / df_final["qty_5yr"]
df_final["ts_volatility_norm"] = df_final["ts_volatility"] / df_final["qty_5yr"]

In [ ]:
df_final["recent_growth"] = df_final["recent_growth"].replace([np.inf, -np.inf], np.nan)
df_final["recent_growth"] = df_final["recent_growth"].fillna(0)

In [ ]:
output_file_path = os.path.join(BASE_PATH, 'data_clean', 'df_final.csv')

df_final.to_csv(output_file_path, index=False)

print(f"df_final successfully saved to {output_file_path}")

# **OCD Dataset**

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
file_path = os.path.join(BASE_PATH, 'data_raw', 'OCD_wells_data.csv')

try:
    df = pd.read_csv(file_path)
    # Convert DataFrame to GeoDataFrame
    geometry = gpd.points_from_xy(df['longitude'], df['latitude'])
    df = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')
    print(f"Successfully loaded data from {file_path}.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
df.shape

In [ ]:
DISPOSAL_WELL_TYPES = ["Salt Water Disposal"] # update: 24/04 -> Injection removed
swd_wells = df[df["type"].isin(DISPOSAL_WELL_TYPES)].copy()

print(f"Total wells in OCD dataset:    {len(df):,}")
print(f"Disposal wells (SWD): {len(swd_wells):,}")
print()
print("Breakdown by type:")
print(swd_wells["type"].value_counts())

In [ ]:
swd_wells["disposal_type"] = swd_wells["type"].map({
    "Salt Water Disposal": "SWD Well",
    "Injection":           "Injection Well"
})


oil_wells = df[df["type"] == "Oil"].copy()
gas_wells = df[df["type"] == "Gas"].copy()

print(f"Oil wells:         {len(oil_wells):,}")
print(f"Gas wells:         {len(gas_wells):,}")
print(f"SWD + Injection wells:         {len(swd_wells):,}")

**visualization**

In [ ]:
import folium

m = folium.Map(location=[34.5, -106.0], zoom_start=7, tiles="CartoDB positron")

# Colour-code by disposal type
colors = {"SWD Well": "#1D9E75", "Injection Well": "#378ADD"}

for _, row in swd_wells.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color=colors.get(row["disposal_type"], "#888"),
        fill=True, fill_opacity=0.7,
        popup=f"{row['disposal_type']}\n{row.get('WellName','')}"
    ).add_to(m)

m

exporting: swd_wells.geojson — all disposal wells (SWD + Injection) with geometry

oil_wells.geojson — oil production wells.

nm_swd_per_county.geojson — county polygons enriched with SWD well counts.

ocd_data_dictionary.md — documents what each dataset is

In [ ]:
import os
import pandas as pd
import geopandas as gpd

# Define paths relative to BASE_PATH for consistency
DATA_DIR = os.path.join(BASE_PATH, 'data_processed', 'ocd')
os.makedirs(DATA_DIR, exist_ok=True)

# Save disposal wells
swd_wells.to_file(
    os.path.join(DATA_DIR, "nm_ocd_disposal_wells.geojson"),
    driver="GeoJSON"
)
print(f"Saved: nm_ocd_disposal_wells.geojson  ({len(swd_wells):,} wells)")

# Save oil wells for cross-validation
oil_wells.to_file(
    os.path.join(DATA_DIR, "nm_ocd_oil_wells.geojson"),
    driver="GeoJSON"
)
print(f"Saved: nm_ocd_oil_wells.geojson        ({len(oil_wells):,} wells)")

# County-level aggregation
# Group swd_wells by county_name and count wells
county_well_counts = swd_wells.groupby('county').size().reset_index(name='swd_well_count')

# For geometry, calculate the centroid of all wells within each county
# This results in point geometries representing the "center" of well activity per county
county_geometries_centroids = swd_wells.groupby('county').apply(
    lambda x: x.geometry.union_all().centroid if not x.geometry.empty else None
).reset_index(name='geometry')

# Merge counts and geometries
nm_swd = pd.merge(county_well_counts, county_geometries_centroids, on='county', how='left')

# Convert to GeoDataFrame
nm_swd = gpd.GeoDataFrame(nm_swd, geometry='geometry', crs=swd_wells.crs)

nm_swd.to_file(
    os.path.join(DATA_DIR, "nm_swd_per_county.geojson"),
    driver="GeoJSON"
)
print(f"Saved: nm_swd_per_county.geojson       ({len(nm_swd)} counties)")

# a CSV summary
swd_summary = (swd_wells
    .drop(columns="geometry")
    .groupby(["county","disposal_type"])
    .size()
    .reset_index(name="count")
)
swd_summary.to_csv(os.path.join(DATA_DIR, "nm_swd_summary.csv"), index=False)
print(f"Saved: nm_swd_summary.csv")